In [1]:
import pandas as pd
import numpy as np
import random
from tqdm import tqdm

In [2]:
articles = pd.read_csv("data/articles.csv")
transactions = pd.read_csv("data/transactions_train.csv")

# format id
articles["article_id"] = articles["article_id"].astype(str).str.zfill(10)
transactions["article_id"] = transactions["article_id"].astype(str).str.zfill(10)

transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

In [3]:
def build_item_sim_dict(articles):
    sim_dict = {}

    # group theo product_type + group
    grouped = articles.groupby(["product_type_name", "product_group_name"])

    for _, group in grouped:
        items = group["article_id"].tolist()

        for item in items:
            # remove itself
            sim_items = [i for i in items if i != item]

            # limit size (tránh quá lớn)
            if len(sim_items) > 50:
                sim_items = random.sample(sim_items, 50)

            sim_dict[item] = sim_items

    return sim_dict

item_sim_dict = build_item_sim_dict(articles)

### Gen Click

In [4]:
def generate_clicks(purchase_item, sim_dict, num_click=5):
    if purchase_item not in sim_dict:
        return []

    sim_items = sim_dict[purchase_item]

    if len(sim_items) == 0:
        return []

    num = min(num_click, len(sim_items))

    return random.sample(sim_items, num)

### Gen Cart

In [5]:
def generate_cart(clicks, purchase_item):
    cart = []

    for item in clicks:
        prob = 0.3

        # nếu cùng category → tăng xác suất
        if item[:3] == purchase_item[:3]:
            prob = 0.6

        if random.random() < prob:
            cart.append(item)

    return cart

## Build full behavior

In [6]:
def build_fake_behavior(transactions, item_sim_dict):
    rows = []

    for _, row in tqdm(transactions.iterrows(), total=len(transactions)):
        user = row["customer_id"]
        item = row["article_id"]
        time = row["t_dat"]

        # ===== PURCHASE (giữ nguyên) =====
        rows.append({
            "t_dat": time,
            "customer_id": user,
            "article_id": item,
            "event_type": "purchase"
        })

        # ===== CLICK =====
        clicks = generate_clicks(item, item_sim_dict)

        for i, c in enumerate(clicks):
            rows.append({
                "t_dat": time - pd.Timedelta(days=3-i),
                "customer_id": user,
                "article_id": c,
                "event_type": "click"
            })

        # ===== CART =====
        carts = generate_cart(clicks, item)

        for i, c in enumerate(carts):
            rows.append({
                "t_dat": time - pd.Timedelta(days=1),
                "customer_id": user,
                "article_id": c,
                "event_type": "cart"
            })

    df = pd.DataFrame(rows)

    # sort timeline
    df = df.sort_values(["customer_id", "t_dat"])

    return df

In [7]:
fake_df = build_fake_behavior(transactions, item_sim_dict)
fake_df

100%|██████████| 280528/280528 [01:17<00:00, 3636.78it/s]


,t_dat,customer_id,article_id,event_type
803336,2020-08-09,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0825642005,click
803343,2020-08-09,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0513699009,click
803351,2020-08-09,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0906794001,click
803359,2020-08-09,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0880312004,click
803337,2020-08-10,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0855793001,click
...,...,...,...,...
1635417,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,0742916003,click
1635424,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,0761696002,click
1635433,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,0796600003,click
1635441,2020-08-14,ffffbbf78b6eaac697a8a5dfbfd2bfa8113ee5b403e474...,0582480024,click


In [8]:
fake_df.to_csv("data/fake_behavior.csv", index=False)